# Hand Segmentation (Layer 2b)

From-scratch small U-Net on FreiHAND with background-paste augmentation.

- Input: 192x192 RGB
- Output: 192x192 binary mask (hand = 1, background = 0)
- Architecture: 4-level U-Net, channels 32->64->128->256, ~2M params
- Loss: BCE + Dice
- Aug: random scale/position/flip/rot/color + random background paste using Places365 indoor subset

**Prereq downloads** (see Cell 3 for instructions):
- FreiHAND v2 → `../data/freihand/`
- Background images → `../data/backgrounds/`

In [ ]:
import os, json, random, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFilter
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## Dataset downloads

**FreiHAND v2** (~4GB):
```bash
cd ../data/freihand
curl -L -o FreiHAND_pub_v2.zip https://lmb.informatik.uni-freiburg.de/data/freihand/FreiHAND_pub_v2.zip
unzip FreiHAND_pub_v2.zip
# Should produce: training/rgb/, training/mask/, evaluation/, etc.
```

**Places365 indoor subset** (~1GB, ~1000 indoor room scenes):
```bash
cd ../data/backgrounds
# Using Places365 small validation set (smallest, easiest).
curl -L -o places365_val.tar http://data.csail.mit.edu/places/places365/val_256.tar
tar -xf places365_val.tar
# Produces val_256/ with 36,500 256x256 images of diverse scenes.
# We just need ~1000-5000, the dataloader will pick randomly.
```

Or use any folder of indoor/scene images you already have — the dataloader just scans recursively for jpg/png.

In [ ]:
# ---- paths ----
FREIHAND_ROOT = Path('../data/freihand/training')
RGB_DIR  = FREIHAND_ROOT / 'rgb'
MASK_DIR = FREIHAND_ROOT / 'mask'
BG_ROOT  = Path('../data/backgrounds')

CKPT_DIR = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / 'hand_seg_best.pt'
LOG_PATH  = CKPT_DIR / 'hand_seg.log.json'

# ---- training config ----
IMG_SIZE = 192
BATCH    = 32
EPOCHS   = 30
LR       = 3e-4
WD       = 1e-4
PATIENCE = 6

# FreiHAND has 32560 base green-screen images at indices 0-32559
# (130240 total but indices 32560+ are just bg-composited versions of the same hand).
# We use only the base set and paste our own backgrounds.
N_BASE = 32560

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

assert RGB_DIR.is_dir(),  f'missing {RGB_DIR} — run download in prev cell'
assert MASK_DIR.is_dir(), f'missing {MASK_DIR}'
assert BG_ROOT.is_dir(),  f'missing {BG_ROOT}'
print('paths ok')

In [ ]:
# Scan background images recursively.
BG_EXTS = {'.jpg','.jpeg','.png','.JPG','.JPEG','.PNG'}
bg_paths = [p for p in BG_ROOT.rglob('*') if p.suffix in BG_EXTS]
print(f'backgrounds found: {len(bg_paths)}')
assert len(bg_paths) >= 50, 'need at least ~50 background images for meaningful variety'

# Verify FreiHAND indices.
_sample_rgb  = RGB_DIR  / '00000000.jpg'
_sample_mask = MASK_DIR / '00000000.jpg'
assert _sample_rgb.is_file()  and _sample_mask.is_file(), 'FreiHAND files not found at expected names'
print('freihand ok')

In [ ]:
# Random index split (FreiHAND doesn't have per-subject ids in the public release).
all_idx = list(range(N_BASE))
rng = random.Random(SEED); rng.shuffle(all_idx)
n_tr = int(N_BASE * 0.90)
n_va = int(N_BASE * 0.05)
train_idx = all_idx[:n_tr]
val_idx   = all_idx[n_tr:n_tr+n_va]
test_idx  = all_idx[n_tr+n_va:]
print(f'split: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}')

In [ ]:
class HandSegDataset(Dataset):
    """FreiHAND + background-paste aug.

    __getitem__ returns: (image_tensor [3,H,W] normalized, mask_tensor [1,H,W] in {0,1}).
    """
    def __init__(self, indices, bg_paths, img_size, mode='train'):
        assert mode in ('train','eval')
        self.indices = indices
        self.bg_paths = bg_paths
        self.img_size = img_size
        self.mode = mode

    def __len__(self): return len(self.indices)

    def _load(self, i):
        rgb  = Image.open(RGB_DIR  / f'{i:08d}.jpg').convert('RGB')
        mask = Image.open(MASK_DIR / f'{i:08d}.jpg').convert('L')
        mask = mask.point(lambda v: 255 if v > 127 else 0)
        mask = mask.filter(ImageFilter.MinFilter(3))
        return rgb, mask

    def _skin_tone(self, rgb, mask):
        """Shift hand pixels darker to cover skin tones FreiHAND under-represents.
        Only darkens (cap=1.0); lightening past the default produces unnatural pink/magenta.
        At darker factors, also desaturate toward brown: real darker skin is less pink
        than just 'dimmed FreiHAND hand', which would otherwise bake a pink bias in."""
        arr = np.array(rgb, dtype=np.float32)
        m = (np.array(mask, dtype=np.float32) / 255.0)[..., None]  # [H,W,1]
        factor = random.uniform(0.35, 1.0)
        # blend toward grey (desaturate) as hand gets darker
        desat = 1.0 - 0.4 * (1.0 - factor)   # factor 1.0 -> desat 1.0 (no change); factor 0.35 -> desat 0.74
        mean = arr.mean(axis=-1, keepdims=True)
        recolored = (arr * desat + mean * (1 - desat)) * factor
        arr = arr * (1.0 - m) + recolored * m
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    def _paste_bg(self, rgb, mask):
        bg_path = random.choice(self.bg_paths)
        try:
            bg = Image.open(bg_path).convert('RGB')
        except Exception:
            return rgb
        W, H = rgb.size
        bw, bh = bg.size
        scale = max(W/bw, H/bh) * random.uniform(1.0, 1.5)
        nbw, nbh = int(bw*scale), int(bh*scale)
        bg = bg.resize((nbw, nbh), Image.BILINEAR)
        x0 = random.randint(0, max(0, nbw - W))
        y0 = random.randint(0, max(0, nbh - H))
        bg = bg.crop((x0, y0, x0 + W, y0 + H))
        if random.random() < 0.3:
            bg = bg.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 2.0)))
        return Image.composite(rgb, bg, mask)

    def _scale_place(self, rgb, mask):
        target = self.img_size
        s = random.uniform(0.3, 1.0)
        new_w = new_h = max(8, int(target*s))
        rgb_s  = rgb.resize((new_w, new_h), Image.BILINEAR)
        mask_s = mask.resize((new_w, new_h), Image.NEAREST)
        bg_path = random.choice(self.bg_paths)
        try:
            canvas = Image.open(bg_path).convert('RGB').resize((target, target), Image.BILINEAR)
        except Exception:
            canvas = Image.new('RGB', (target, target), (127,127,127))
        mask_canvas = Image.new('L', (target, target), 0)
        x = random.randint(0, target - new_w)
        y = random.randint(0, target - new_h)
        canvas.paste(rgb_s, (x, y), mask_s)
        mask_canvas.paste(mask_s, (x, y))
        return canvas, mask_canvas

    def _augment(self, rgb, mask):
        rgb = self._skin_tone(rgb, mask)

        if random.random() < 0.6:
            rgb, mask = self._scale_place(rgb, mask)
        else:
            rgb = self._paste_bg(rgb, mask)
            rgb  = rgb.resize((self.img_size, self.img_size),  Image.BILINEAR)
            mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

        if random.random() < 0.5:
            rgb  = TF.hflip(rgb); mask = TF.hflip(mask)
        angle = random.uniform(-20, 20)
        rgb  = TF.rotate(rgb,  angle, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)
        rgb = TF.adjust_brightness(rgb, random.uniform(0.8, 1.2))
        rgb = TF.adjust_contrast(rgb,   random.uniform(0.8, 1.2))
        rgb = TF.adjust_saturation(rgb, random.uniform(0.8, 1.1))  # narrower so we don't undo skin-tone desat
        rgb = TF.adjust_hue(rgb,        random.uniform(-0.04, 0.04))
        return rgb, mask

    def __getitem__(self, k):
        i = self.indices[k]
        rgb, mask = self._load(i)
        if self.mode == 'train':
            rgb, mask = self._augment(rgb, mask)
        else:
            rgb, mask = self._scale_place(rgb, mask)

        rgb  = rgb.resize((self.img_size, self.img_size),  Image.BILINEAR)
        mask = mask.resize((self.img_size, self.img_size), Image.NEAREST)

        x = TF.to_tensor(rgb)
        x = TF.normalize(x, IMAGENET_MEAN, IMAGENET_STD)
        m = TF.to_tensor(mask)
        m = (m > 0.5).float()
        return x, m

In [ ]:
ds = HandSegDataset(train_idx[:16], bg_paths, IMG_SIZE, mode='train')
x, m = ds[0]
print('img:', x.shape, x.dtype, 'mask:', m.shape, m.dtype, 'mask mean:', m.mean().item())
assert tuple(x.shape) == (3, IMG_SIZE, IMG_SIZE)
assert tuple(m.shape) == (1, IMG_SIZE, IMG_SIZE)

In [ ]:
def _denorm(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD ).view(3,1,1)
    return (t*std + mean).clamp(0,1)

# Curated skin-tone palette. Browns/tans only — NO pink. Ordered dark -> light.
SKIN_PALETTE = [
    (58, 49, 42),    # very dark brown
    (96, 65, 52),    # dark brown
    (130, 92, 67),   # medium brown
    (160, 126, 86),  # tan
    (200, 170, 130), # light tan
    (215, 189, 150), # pale warm
    (234, 218, 186), # beige
]

def _silhouette_on_bg(mask_np, bg_np, rng):
    """Fill mask region with a random skin-palette color, composited onto bg.
    mask_np: HxW float in [0,1]. bg_np: HxWx3 float in [0,1]. Returns HxWx3 float in [0,1]."""
    color = np.array(rng.choice(SKIN_PALETTE), dtype=np.float32) / 255.0  # [3]
    m = mask_np[..., None]  # HxWx1
    fill = np.broadcast_to(color, bg_np.shape)
    return bg_np * (1 - m) + fill * m

# Load a few backgrounds for the overlay viz
_rng = random.Random(0)
_bg_samples = [np.array(Image.open(_rng.choice(bg_paths)).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0 for _ in range(6)]

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for i in range(6):
    x, m = ds[i % len(ds)]
    img  = _denorm(x).permute(1,2,0).numpy()
    mask = m[0].numpy()
    overlay = _silhouette_on_bg(mask, _bg_samples[i], _rng)
    axes[0,i].imshow(img);     axes[0,i].set_title('aug sample'); axes[0,i].axis('off')
    axes[1,i].imshow(mask, cmap='gray'); axes[1,i].set_title('mask'); axes[1,i].axis('off')
    axes[2,i].imshow(overlay); axes[2,i].set_title('silhouette on bg'); axes[2,i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
def conv_block(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
    )

class HandUNet(nn.Module):
    """Small 4-level U-Net. Channels 32->64->128->256, ~2M params."""
    def __init__(self):
        super().__init__()
        self.d1 = conv_block(3,   32)
        self.d2 = conv_block(32,  64)
        self.d3 = conv_block(64,  128)
        self.d4 = conv_block(128, 256)   # bottleneck
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.u3  = conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128,  64, 2, stride=2)
        self.u2  = conv_block(128,  64)
        self.up1 = nn.ConvTranspose2d(64,   32, 2, stride=2)
        self.u1  = conv_block(64,   32)
        self.out = nn.Conv2d(32, 1, 1)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        c1 = self.d1(x)               # 192
        c2 = self.d2(self.pool(c1))   # 96
        c3 = self.d3(self.pool(c2))   # 48
        c4 = self.d4(self.pool(c3))   # 24 bottleneck
        u3 = self.u3(torch.cat([self.up3(c4), c3], dim=1))  # 48
        u2 = self.u2(torch.cat([self.up2(u3), c2], dim=1))  # 96
        u1 = self.u1(torch.cat([self.up1(u2), c1], dim=1))  # 192
        return self.out(u1)   # logits [B,1,H,W]

model = HandUNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'params: {n_params:,}')
with torch.no_grad():
    _y = model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device))
print('output:', _y.shape)
assert tuple(_y.shape) == (2, 1, IMG_SIZE, IMG_SIZE)

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    p = torch.sigmoid(logits)
    num = 2 * (p * targets).sum(dim=(2,3)) + eps
    den = p.sum(dim=(2,3)) + targets.sum(dim=(2,3)) + eps
    return 1 - (num / den).mean()

bce = nn.BCEWithLogitsLoss()

def seg_loss(logits, targets):
    return bce(logits, targets) + dice_loss(logits, targets)

def iou_score(logits, targets, thr=0.5, eps=1e-6):
    p = (torch.sigmoid(logits) > thr).float()
    inter = (p * targets).sum(dim=(2,3))
    union = ((p + targets) > 0).float().sum(dim=(2,3))
    return ((inter + eps) / (union + eps)).mean().item()

In [ ]:
train_ds = HandSegDataset(train_idx, bg_paths, IMG_SIZE, mode='train')
val_ds   = HandSegDataset(val_idx,   bg_paths, IMG_SIZE, mode='eval')
test_ds  = HandSegDataset(test_idx,  bg_paths, IMG_SIZE, mode='eval')

# num_workers=0 in notebook because spawn can't pickle notebook-defined classes on macOS.
# Use train.py (with proper __main__ guard) for num_workers>0.
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0)

xb, mb = next(iter(train_loader))
print('batch:', xb.shape, mb.shape)

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, loader, desc='val'):
    model.eval()
    tot_loss, tot_iou, n = 0.0, 0.0, 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for x, m in pbar:
        x, m = x.to(device), m.to(device)
        logits = model(x)
        loss = seg_loss(logits, m)
        iou  = iou_score(logits, m)
        tot_loss += loss.item() * x.size(0)
        tot_iou  += iou * x.size(0)
        n += x.size(0)
        pbar.set_postfix(loss=f'{tot_loss/n:.4f}', iou=f'{tot_iou/n:.4f}')
    return {'loss': tot_loss/n, 'iou': tot_iou/n}

def train_one_epoch(model, loader, opt, desc='train'):
    model.train()
    tot_loss, n = 0.0, 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for x, m in pbar:
        x, m = x.to(device), m.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = seg_loss(logits, m)
        loss.backward(); opt.step()
        tot_loss += loss.item() * x.size(0); n += x.size(0)
        pbar.set_postfix(loss=f'{tot_loss/n:.4f}')
    return tot_loss/n

In [ ]:
_opt = torch.optim.Adam(model.parameters(), lr=LR)
_x, _m = next(iter(train_loader))
_x, _m = _x.to(device), _m.to(device)
_opt.zero_grad(); _l = seg_loss(model(_x), _m); _l.backward(); _opt.step()
print('smoke loss:', _l.item())

## Training loop

Reset model, run the full training. Use `train.py` for `num_workers>0` speed.

In [ ]:
model = HandUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_iou = -1.0
no_improve = 0

print(f'device={device} | params={n_params:,} | batches/ep: train={len(train_loader)} val={len(val_loader)}', flush=True)

t0 = time.time()
for ep in range(EPOCHS):
    print(f'\n=== epoch {ep}/{EPOCHS-1} ===', flush=True)
    tr_loss = train_one_epoch(model, train_loader, optimizer, desc=f'ep{ep} train')
    val = evaluate(model, val_loader, desc=f'ep{ep} val')
    scheduler.step()

    improved = val['iou'] > best_iou
    if improved:
        best_iou = val['iou']; no_improve = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'img_size': IMG_SIZE,
            'normalize_mean': IMAGENET_MEAN,
            'normalize_std':  IMAGENET_STD,
            'val_iou': best_iou,
            'epoch': ep,
        }, CKPT_PATH)
    else:
        no_improve += 1

    history.append({'epoch': ep, 'tr_loss': tr_loss, **val})
    flag = ' *NEW BEST*' if improved else ''
    lr_now = optimizer.param_groups[0]['lr']
    print(f"ep {ep:2d} | tr_loss {tr_loss:.4f} | val_loss {val['loss']:.4f} | val_iou {val['iou']:.4f}{flag} | lr {lr_now:.2e} | elapsed {time.time()-t0:.0f}s", flush=True)

    if no_improve >= PATIENCE:
        print(f'early stop at epoch {ep} (no improvement in {PATIENCE} epochs)', flush=True); break

with open(LOG_PATH, 'w') as f:
    json.dump({'config': {'img_size': IMG_SIZE, 'batch': BATCH, 'epochs': EPOCHS, 'lr': LR, 'wd': WD},
               'history': history, 'best_val_iou': best_iou, 'params': n_params}, f, indent=2)
print(f'\ndone. best_val_iou={best_iou:.4f}')

## Evaluation + qualitative viz

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"loaded ep {ckpt['epoch']} val_iou {ckpt['val_iou']:.4f}")

test = evaluate(model, test_loader)
print(f"TEST | loss {test['loss']:.4f} | iou {test['iou']:.4f}")

In [ ]:
model.eval()
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
with torch.no_grad():
    for i in range(6):
        x, m = test_ds[i]
        logits = model(x.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logits)[0,0].cpu().numpy() > 0.5).astype(float)
        img  = _denorm(x).permute(1,2,0).numpy()
        axes[0,i].imshow(img);        axes[0,i].set_title('image');     axes[0,i].axis('off')
        axes[1,i].imshow(m[0], cmap='gray'); axes[1,i].set_title('ground truth'); axes[1,i].axis('off')
        axes[2,i].imshow(pred, cmap='gray'); axes[2,i].set_title('pred mask');    axes[2,i].axis('off')
plt.tight_layout(); plt.show()